<a href="https://colab.research.google.com/github/PilliSrilekha1419/ai-mentor-portfolio/blob/main/Day10_Sprint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "numpy<2.0"

!pip install -q --upgrade \
crewai \
litellm \
google-generativeai \
google-genai \
chromadb \
sentence-transformers \
crewai-tools

In [ ]:
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

print("API key set")

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool as crewai_tool

from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

import json
import pathlib

In [ ]:
llm = "gemini/gemini-2.5-flash"

print(llm)

In [ ]:
client_db = PersistentClient(path='./chroma_db')

col = client_db.get_or_create_collection('placement_kb')

embed = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

print("Vector DB ready")

In [ ]:
@crewai_tool
def rag_search(query: str) -> str:
    """
    Search placement knowledge base.
    """

    qv = embed.encode([query]).tolist()

    results = col.query(
        query_embeddings=qv,
        n_results=4
    )

    docs = results['documents'][0]

    return '\n---\n'.join(docs)

print("RAG tool created")

In [ ]:
researcher = Agent(
    role="Placement Researcher",

    goal="Research company placement requirements for students",

    backstory="You prepare factual placement research notes using RAG search.",

    llm=llm,

    tools=[rag_search],

    verbose=True,

    allow_delegation=False,
)

print("Researcher agent ready")

In [ ]:
interviewer = Agent(
    role="Mock Interviewer",

    goal="Generate placement interview questions",

    backstory="You generate technical and HR interview questions.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

print("Interviewer agent ready")

In [ ]:
coach = Agent(
    role="Answer Coach",

    goal="Create strong sample answers and preparation guidance",

    backstory="You coach students with strong placement answers.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

print("Coach agent ready")

In [ ]:
tracker = Agent(
    role="Progress Tracker",

    goal="Create structured student progress summaries",

    backstory="You generate JSON-style student progress summaries.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

print("Tracker agent ready")

In [ ]:
profiles = [
    {
        "student_id": "S001",
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 7.8,
        "skills": ["Python", "Java", "SQL"],
        "target_company": "TCS Digital"
    },

    {
        "student_id": "S002",
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 8.1,
        "skills": ["Python", "DBMS"],
        "target_company": "Cognizant"
    },

    {
        "student_id": "S003",
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 8.5,
        "skills": ["Java", "DSA", "OOPs"],
        "target_company": "Amazon"
    }
]

print("Profiles loaded")

In [ ]:
def build_tasks(profile):

    research = Task(
        description=(
            f"Research placement preparation details for "
            f"{profile['target_company']} "
            f"for a {profile['branch']} student."
        ),

        agent=researcher,

        expected_output="Short bullet research notes."
    )

    interview = Task(
        description=(
            f"Generate EXACTLY 10 mock interview questions "
            f"for {profile['name']} targeting "
            f"{profile['target_company']}."
        ),

        agent=interviewer,

        expected_output="Exactly 10 numbered interview questions."
    )

    coaching = Task(
        description=(
            f"Pick question 3 and create strong sample answer "
            f"for {profile['name']}."
        ),

        agent=coach,

        expected_output="One strong answer and 3 preparation tips."
    )

    tracking = Task(
        description=(
            f"Create JSON-style progress summary "
            f"for {profile['student_id']}."
        ),

        agent=tracker,

        expected_output="Valid JSON-style summary."
    )

    return [research, interview, coaching, tracking]

print("Task builder ready")

In [ ]:
p = profiles[0]

crew = Crew(
    agents=[
        researcher,
        interviewer,
        coach,
        tracker
    ],

    tasks=build_tasks(p),

    process=Process.sequential,

    verbose=True,
)

print("Crew created")

In [ ]:
import asyncio

async def run_crew_safe():

    try:
        result = await crew.kickoff_async()

        print("\n=== FINAL OUTPUT ===\n")
        print(result)

        return result

    except Exception as e:

        print("\n❌ Crew Execution Failed")
        print("Error Type:", type(e).__name__)
        print("Error Message:", str(e)[:300])

        print("\n💡 Fix Suggestions:")
        print("1. Gemini quota exceeded (429 error)")
        print("2. Switch to Ollama or Groq")
        print("3. Wait 30–60 seconds and retry")

        return None


# ✅ RUN IN COLAB
result = await run_crew_safe()

In [ ]:
import asyncio

transcripts = []

async def run_safe_crew(p):

    try:
        print("\n" + "="*60)
        print(f"Running for {p['name']} → {p['target_company']}")
        print("="*60)

        crew = Crew(
            agents=[researcher, interviewer, coach, tracker],
            tasks=build_tasks(p),
            process=Process.sequential,
            verbose=True,
        )

        result = await crew.kickoff_async()

        return {
            "student": p["name"],
            "target": p["target_company"],
            "final_output": str(result),
            "status": "success"
        }

    except Exception as e:

        print("\n❌ Crew failed for:", p['name'])
        print("Error:", str(e)[:250])

        # ✅ async-safe delay (IMPORTANT FIX)
        print("⏳ Waiting 30 seconds before next student...\n")
        await asyncio.sleep(30)

        return {
            "student": p["name"],
            "target": p["target_company"],
            "final_output": f"FAILED: {str(e)[:200]}",
            "status": "failed"
        }


# =========================
# MAIN LOOP (SAFE EXECUTION)
# =========================

for p in profiles:

    result = await run_safe_crew(p)
    transcripts.append(result)

print("\nCompleted:", len(transcripts))

In [ ]:
import json
from pathlib import Path

# convert safely (handles CrewAI objects too)
safe_transcripts = json.loads(json.dumps(transcripts, default=str))

Path("day10_sprint5_transcripts.json").write_text(
    json.dumps(safe_transcripts, indent=2),
    encoding="utf-8"
)

print("Saved transcripts successfully")

In [ ]:
from pathlib import Path

md = "# Day10 Sprint5 Report\n\n"

for t in transcripts:

    md += f"## {t['student']} → {t['target']}\n\n"

    # safely convert output to string
    md += str(t["final_output"])

    md += "\n\n---\n\n"

# save markdown file
Path("day10_sprint5_report.md").write_text(
    md,
    encoding="utf-8"
)

print("Markdown report created successfully")

In [ ]:
from google.colab import files

files.download("day10_sprint5_transcripts.json")

files.download("day10_sprint5_report.md")